# [16.2] KernelSHAP and PartitionSHAP Controls - Exercises

Implement full-table KernelSHAP and PartitionSHAP/Owen-value controls, then run the visible tests.

In [ ]:
import itertools
import math
import sys
from collections.abc import Callable, Mapping
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part2_kernelshap_partition_shap_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_kernelshap_partition_shap_controls.tests as tests

Coalition = frozenset[int]
GT_TIER = "GT-0"
EXERCISE_ID = "16.2.kernelshap_partition_shap_controls"
EXPECTED_RUNTIME = "45-70 minutes for exercises; under 2 minutes for the CUDA finite-game report"
REQUIRES_GPU = True


In [ ]:
@dataclass(frozen=True)
class KernelSHAPApproximationReport:
    shapley_values: t.Tensor
    exact_values: t.Tensor
    max_abs_error: float
    approximates_exact: bool


@dataclass(frozen=True)
class PartitionSHAPReport:
    group_values: t.Tensor
    player_values: t.Tensor
    exact_values: t.Tensor
    max_abs_error: float
    recovers_exact: bool


In [ ]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    if num_players <= 0:
        raise ValueError("num_players must be positive.")
    players = range(num_players)
    coalitions = []
    for size in range(num_players + 1):
        coalitions.extend(frozenset(group) for group in itertools.combinations(players, size))
    return tuple(coalitions)


def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    values = {frozenset(key): float(value) for key, value in coalition_values.items()}
    expected = set(all_coalitions(num_players))
    missing = expected - set(values)
    if missing:
        raise ValueError(f"coalition value table is missing {len(missing)} coalitions.")
    return values


def coalition_values_from_function(
    num_players: int,
    value_fn: Callable[[Coalition], float],
) -> dict[Coalition, float]:
    return {coalition: float(value_fn(coalition)) for coalition in all_coalitions(num_players)}


def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    values = normalize_coalition_values(coalition_values, num_players=num_players)
    denominator = math.factorial(num_players)
    shapley = t.zeros(num_players, dtype=t.float64)
    for player in range(num_players):
        others = [item for item in range(num_players) if item != player]
        for size in range(num_players):
            weight = (
                math.factorial(size)
                * math.factorial(num_players - size - 1)
                / denominator
            )
            for group in itertools.combinations(others, size):
                coalition = frozenset(group)
                shapley[player] += weight * (
                    values[coalition | {player}] - values[coalition]
                )
    return shapley


def additive_game(weights: t.Tensor) -> dict[Coalition, float]:
    weights = weights.flatten().double()
    return coalition_values_from_function(
        int(weights.numel()),
        lambda coalition: weights[list(coalition)].sum().item() if coalition else 0.0,
    )


def conjunction_game(num_players: int) -> dict[Coalition, float]:
    full = frozenset(range(num_players))
    return coalition_values_from_function(num_players, lambda coalition: coalition == full)


In [ ]:
def kernelshap_kernel_weight(coalition_size: int, num_players: int) -> float:
    raise NotImplementedError()


tests.test_kernelshap_kernel_weight_uses_finite_coalition_formula(
    kernelshap_kernel_weight,
)


In [ ]:
def kernelshap_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    raise NotImplementedError()


def kernelshap_approximation_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    tolerance: float = 1e-9,
) -> KernelSHAPApproximationReport:
    raise NotImplementedError()


tests.test_kernelshap_approximation_report_matches_exact_additive_game(
    kernelshap_approximation_report,
    additive_game,
)


In [ ]:
tests.test_kernelshap_interaction_report_splits_conjunction_credit(
    kernelshap_approximation_report,
    conjunction_game,
)


In [ ]:
def grouped_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    groups: tuple[tuple[int, ...], ...],
) -> dict[Coalition, float]:
    raise NotImplementedError()


def partition_shap_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    groups: tuple[tuple[int, ...], ...],
) -> t.Tensor:
    raise NotImplementedError()


def partition_shap_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    groups: tuple[tuple[int, ...], ...],
    tolerance: float = 1e-9,
) -> PartitionSHAPReport:
    raise NotImplementedError()


tests.test_partition_shap_report_recovers_additive_groups(
    partition_shap_report,
    additive_game,
)
tests.test_partition_shap_report_rejects_invalid_grouping(
    partition_shap_report,
    additive_game,
)


In [ ]:
tests.test_partition_shap_report_splits_interaction_group_symmetrically(
    partition_shap_report,
    conjunction_game,
)


In [ ]:
def _tensor_report(report: object) -> dict:
    result = report.__dict__.copy()
    for key, value in list(result.items()):
        if hasattr(value, "tolist"):
            result[key] = value.tolist()
    return result


def kernel_additive_smoke_test() -> dict:
    values = additive_game(t.tensor([1.0, -2.0, 0.5]))
    return _tensor_report(kernelshap_approximation_report(values, num_players=3))


def kernel_interaction_smoke_test() -> dict:
    values = conjunction_game(3)
    return _tensor_report(kernelshap_approximation_report(values, num_players=3))


def partition_additive_smoke_test() -> dict:
    values = additive_game(t.tensor([1.0, 2.0, 3.0, 4.0]))
    return _tensor_report(partition_shap_report(values, groups=((0, 1), (2, 3))))


def partition_interaction_smoke_test() -> dict:
    values = conjunction_game(2)
    return _tensor_report(partition_shap_report(values, groups=((0, 1),)))


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "kernel_additive": kernel_additive_smoke_test(),
        "kernel_interaction": kernel_interaction_smoke_test(),
        "partition_additive": partition_additive_smoke_test(),
        "partition_interaction": partition_interaction_smoke_test(),
    }


tests.test_notebook_contract(run_smoke_test)


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
